# 1.1 ETL Pipeline
Extracts raw SimFin data, computes technical features, adds the target variable, and saves one processed CSV per ticker to `../data/processed/`.

In [30]:
import os
import numpy as np
import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────
# Edit INPUT_PATH to where you put the SimFin bulk download files
INPUT_PATH = "us-shareprices-daily.csv"
OUTPUT_DIR = "../data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load raw share prices

In [31]:
df_share_prices = pd.read_csv(INPUT_PATH, delimiter=";")
print("Shape:", df_share_prices.shape)
df_share_prices.head()

Shape: (6205521, 11)


,Ticker,SimFinId,Date,Open,High,Low,Close,Adj. Close,Volume,Dividend,Shares Outstanding
0,A,45846,2020-04-07,76.61,77.56,74.02,74.03,71.17,2458009,NaN,309651359.0
1,A,45846,2020-04-08,74.17,77.17,72.75,76.69,73.73,2702954,NaN,309651359.0
2,A,45846,2020-04-09,76.43,78.72,76.23,78.33,75.31,2399863,NaN,309651359.0
3,A,45846,2020-04-13,77.44,77.99,75.02,76.21,73.27,1533000,NaN,309651359.0
4,A,45846,2020-04-14,77.30,79.20,77.24,78.83,75.79,2650262,NaN,309651359.0


## Load companies list

In [32]:
df_companies = pd.read_csv("us-companies.csv", delimiter=";")
print("Shape:", df_companies.shape)
# Sort by number of employees to get a sense of the biggest companies
df_companies.sort_values("Number Employees", ascending=False).head(10)

Shape: (6531, 11)


,Ticker,SimFinId,Company Name,IndustryId,ISIN,End of financial year (month),Number Employees,Business Summary,Market,CIK,Main Currency
399,AMZN,62747,AMAZON COM INC,103002.0,US0231351067,12.0,1298000.0,Amazon.com Inc is an online retailer. The Comp...,us,1018724.0,USD
6008,UPS,106423,UNITED PARCEL SERVICE INC,100010.0,US9113121068,12.0,543000.0,United Parcel Service Inc is a package deliver...,us,1090727.0,USD
120,ACN,61372,Accenture plc,108002.0,IE00B4BNMY34,8.0,506000.0,Accenture PLC is a professional service compan...,us,1467373.0,USD
2827,HOME,233408,At Home Group Inc.,103002.0,US04650Y1001,1.0,400000.0,At Home Group Inc is a home décor superstore. ...,us,1646228.0,USD
5692,TESS,42813,TESSCO TECHNOLOGIES INC,101005.0,US8723861071,3.0,391500.0,Tessco Technologies Inc architects and deliver...,us,927355.0,USD
961,BRK-A,71306,BERKSHIRE HATHAWAY INC,111001.0,US0846701086,12.0,371653.0,"Berkshire Hathaway Inc., through its subsidiar...",us,1067983.0,USD
2308,FMX,18773575,"Fomento Económico Mexicano, S.A.B. de C.V.",102004.0,US3444191064,12.0,350000.0,"Fomento Económico Mexicano, S.A.B. de C.V., th...",us,1061736.0,USD
5129,SBUX,86689,STARBUCKS CORP,103003.0,US8552441094,9.0,349000.0,"Starbucks Corp is the roaster, marketer and re...",us,829224.0,USD
2944,IBM,69543,INTERNATIONAL BUSINESS MACHINES CORP,101003.0,US4592001014,12.0,345900.0,International Business Machines Corp offers a ...,us,51143.0,USD
5794,TNET,797860,TRINET GROUP INC,100002.0,US8962881079,12.0,334600.0,Trinet Group Inc provides human resources solu...,us,937098.0,USD


## Step 1 — Filter to one company
Develop and test on AMZN first, then scale to all tickers.

In [33]:
def filter_company(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """Filter to one ticker and sort chronologically."""
    company_df = df[df["Ticker"] == ticker].copy()
    company_df.sort_values("Date", ascending=True, inplace=True)
    company_df.reset_index(drop=True, inplace=True)
    return company_df

df_amzn = filter_company(df_share_prices, "AMZN")
print(f"AMZN rows: {len(df_amzn)}")
df_amzn.head()

AMZN rows: 1238


,Ticker,SimFinId,Date,Open,High,Low,Close,Adj. Close,Volume,Dividend,Shares Outstanding
0,AMZN,62747,2020-04-07,100.86,101.79,99.88,100.58,100.58,102279660,NaN,9.980000e+09
1,AMZN,62747,2020-04-08,101.05,102.20,100.56,102.15,102.15,79546260,NaN,9.980000e+09
2,AMZN,62747,2020-04-09,102.22,102.65,100.88,102.14,102.14,93112340,NaN,9.980000e+09
3,AMZN,62747,2020-04-13,102.00,109.00,101.90,108.44,108.44,134334180,NaN,9.980000e+09
4,AMZN,62747,2020-04-14,110.02,114.60,109.31,114.17,114.17,161743860,NaN,9.980000e+09


## Step 2 — Add technical features

In [34]:
def add_technical_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute 8 technical indicator features from raw share price data.
    Drops rows with NaN values caused by rolling windows.
    """
    df = df.copy()

    # Daily log-return
    df["Returns"] = np.log(df["Close"] / df["Close"].shift(1))

    # Simple Moving Averages
    df["SMA_5"]  = df["Close"].rolling(window=5).mean()   # 1 trading week
    df["SMA_20"] = df["Close"].rolling(window=20).mean()  # ~1 trading month

    # Rolling volatility (std of returns)
    df["Volatility_5"]  = df["Returns"].rolling(window=5).std()
    df["Volatility_20"] = df["Returns"].rolling(window=20).std()

    # Volume change (% change day-over-day)
    # Volume spikes often signal news events or strong buying/selling pressure
    df["Volume_Change"] = df["Volume"].pct_change()

    # RSI — 14-day Relative Strength Index
    # Momentum indicator: >70 = overbought, <30 = oversold
    delta    = df["Close"].diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=14).mean()
    avg_loss = loss.rolling(window=14).mean()
    rs       = avg_gain / avg_loss
    df["RSI_14"] = 100 - (100 / (1 + rs))

    # Normalised price range — intraday volatility relative to price
    # Dividing by Close allows fair comparison across cheap and expensive stocks
    df["Price_Range"] = (df["High"] - df["Low"]) / df["Close"]

    # Drop rows where any feature is NaN (first ~20 rows due to rolling windows)
    feature_cols = [
        "Returns", "SMA_5", "SMA_20",
        "Volatility_5", "Volatility_20",
        "Volume_Change", "RSI_14", "Price_Range"
    ]
    df = df.dropna(subset=feature_cols).reset_index(drop=True)
    return df

df_amzn = add_technical_features(df_amzn)
print(f"Shape after features: {df_amzn.shape}")
df_amzn[["Date","Close","Returns","SMA_5","RSI_14","Price_Range"]].head()

Shape after features: (1218, 19)


,Date,Close,Returns,SMA_5,RSI_14,Price_Range
0,2020-05-06,117.56,0.014307,117.450,45.977985,0.015907
1,2020-05-07,118.38,0.006951,116.386,49.465163,0.013854
2,2020-05-08,118.98,0.005056,117.322,48.978400,0.012691
3,2020-05-11,120.45,0.012279,118.252,56.223044,0.019676
4,2020-05-12,117.85,-0.021822,118.644,49.519520,0.027153


## Step 3 — Add target variable
`Target = 1` if tomorrow's Close > today's Close, else `0`.
This is the binary label the ML model will learn to predict.

In [35]:
def add_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create binary target: 1 if tomorrow Close > today Close, else 0.
    The last row gets NaN (no tomorrow) — drop it before training.
    """
    df = df.copy()
    df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(float)
    return df

df_amzn = add_target(df_amzn)
print("Target distribution:")
print(df_amzn["Target"].value_counts())
df_amzn[["Date","Close","Target"]].tail(5)

Target distribution:
Target
1.0    628
0.0    590
Name: count, dtype: int64


,Date,Close,Target
1213,2025-03-05,208.36,0.0
1214,2025-03-06,200.70,0.0
1215,2025-03-07,199.25,0.0
1216,2025-03-10,194.54,1.0
1217,2025-03-11,196.59,0.0


## Step 4 — Drop columns with many NaN values
Removes columns where more than 75% of values are missing (e.g. Dividend).

In [36]:
def drop_columns_many_nans(df: pd.DataFrame, threshold: float = 0.75) -> pd.DataFrame:
    """Drop columns where the fraction of NaN values exceeds the threshold."""
    return df.loc[:, df.isna().mean() < threshold]

df_amzn = drop_columns_many_nans(df_amzn, threshold=0.75)
print("Remaining columns:", df_amzn.columns.tolist())
print("Any remaining NaN:", df_amzn.isnull().sum().sum())

Remaining columns: ['Ticker', 'SimFinId', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj. Close', 'Volume', 'Shares Outstanding', 'Returns', 'SMA_5', 'SMA_20', 'Volatility_5', 'Volatility_20', 'Volume_Change', 'RSI_14', 'Price_Range', 'Target']
Any remaining NaN: 0


## Step 5 — Save processed file

In [37]:
def save_processed(df: pd.DataFrame, output_dir: str, ticker: str) -> str:
    """Save the processed DataFrame to CSV. Returns the file path."""
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, f"{ticker.lower()}_processed.csv")
    df.to_csv(path, index=False)
    print(f"[ETL] Saved {ticker}: {len(df)} rows → {path}")
    return path

save_processed(df_amzn, OUTPUT_DIR, "AMZN")

[ETL] Saved AMZN: 1218 rows → ../data/processed/amzn_processed.csv


'../data/processed/amzn_processed.csv'

## Step 6 — Full pipeline function
Orchestrates all steps above into one call.

In [38]:
def extract_share_prices(input_path: str) -> pd.DataFrame:
    """Load the raw SimFin share-prices CSV."""
    return pd.read_csv(input_path, delimiter=";")


def run_etl(input_path: str, output_dir: str, ticker: str) -> pd.DataFrame:
    """
    Execute the full ETL pipeline for a single ticker.

    Parameters
    ----------
    input_path : path to us-shareprices-daily.csv
    output_dir : where to save the processed CSV
    ticker     : e.g. 'AMZN'

    Returns
    -------
    Processed DataFrame (also saved to disk).
    """
    raw = extract_share_prices(input_path)
    df  = filter_company(raw, ticker)

    if df.empty:
        raise ValueError(f"No data found for ticker '{ticker}'.")

    df = add_technical_features(df)
    df = add_target(df)
    df = drop_columns_many_nans(df, threshold=0.75)

    save_processed(df, output_dir, ticker)
    return df

## Step 7 — Run ETL for all 5 tickers
Change this list to match whichever companies your group chose.

In [41]:
TICKERS = ["AMZN", "AAPL", "MSFT", "GOOG", "TSLA"]

for ticker in TICKERS:
    run_etl(INPUT_PATH, OUTPUT_DIR, ticker)

[ETL] Saved AMZN: 1218 rows → ../data/processed/amzn_processed.csv
[ETL] Saved AAPL: 1218 rows → ../data/processed/aapl_processed.csv
[ETL] Saved MSFT: 1218 rows → ../data/processed/msft_processed.csv
[ETL] Saved GOOG: 1218 rows → ../data/processed/goog_processed.csv
[ETL] Saved TSLA: 1218 rows → ../data/processed/tsla_processed.csv


## Quick sanity check — inspect one processed file

In [42]:
df_check = pd.read_csv(f"{OUTPUT_DIR}/amzn_processed.csv")
print("Shape:", df_check.shape)
print("Columns:", df_check.columns.tolist())
print("\nTarget balance:")
print(df_check["Target"].value_counts())
df_check.tail(3)

Shape: (1218, 19)
Columns: ['Ticker', 'SimFinId', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj. Close', 'Volume', 'Shares Outstanding', 'Returns', 'SMA_5', 'SMA_20', 'Volatility_5', 'Volatility_20', 'Volume_Change', 'RSI_14', 'Price_Range', 'Target']

Target balance:
Target
1.0    628
0.0    590
Name: count, dtype: int64


,Ticker,SimFinId,Date,Open,High,Low,Close,Adj. Close,Volume,Shares Outstanding,Returns,SMA_5,SMA_20,Volatility_5,Volatility_20,Volume_Change,RSI_14,Price_Range,Target
1215,AMZN,62747,2025-03-07,199.49,202.27,192.53,199.25,199.25,59802821,1.059773e+10,-0.007251,203.426,217.6890,0.024440,0.018462,0.199324,19.914128,0.048883,0.0
1216,AMZN,62747,2025-03-10,195.60,196.73,190.85,194.54,194.54,61829231,1.059773e+10,-0.023923,201.330,215.9585,0.022375,0.017225,0.033885,18.879628,0.030225,1.0
1217,AMZN,62747,2025-03-11,193.90,200.18,193.40,196.59,196.59,54002880,1.059773e+10,0.010483,199.888,214.1310,0.024331,0.016758,-0.126580,21.988064,0.034488,0.0
